In [71]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from sqlalchemy import create_engine
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

In [30]:
server = r'DESKTOP-LEVIKVU\SQLEXPRESS'
database = 'Airbnb'

connection_url = (
    f"mssql+pyodbc://@{server}/{database}?"
    "driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes"
)

engine = create_engine(connection_url)

#read files from sql server
reviews = pd.read_sql(
    'SELECT * FROM reviews',engine
)
listing = pd.read_sql(
    'SELECT * FROM listing',engine
)

#calender = pd.read_sql(
#    'SELECT * FROM calender',engine
#)


In [52]:
reviews = reviews.head()

In [61]:
reviews

,listing_id,id,date,reviewer_id,reviewer_name,comments,city,unique_listing_id
0,839610,33387684,2015-05-27,32412055,Giuseppe,Nice time in nice place! Close to city center!...,Amsterdam,Amsterdam_839610
1,839610,33865462,2015-06-01,33521461,Chi Kwan,The host is very friendly and helpful,Amsterdam,Amsterdam_839610
2,839610,47062612,2015-09-15,25521258,Scott,"Michael, Jacob's son was the one to coordinate...",Amsterdam,Amsterdam_839610
3,839610,49847491,2015-10-06,32115691,Omer,Our stay was excelent. The appartment was clea...,Amsterdam,Amsterdam_839610
4,839610,62280485,2016-02-13,16551781,Nina,it's very good! thanks! it's very beautiful al...,Amsterdam,Amsterdam_839610


In [42]:

MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [46]:
#Roberta RESULTS
print(example)

# Changed to return_tensors
encoded_text = tokenizer(example, return_tensors='pt') 

output = model(**encoded_text)
scores = output[0][0].detach().numpy()
scores = softmax(scores) 
scores_dict ={'roberta_neg':scores[0],'roberta_neu':scores[1],'roberta_pos':scores[2]}
scores_dict

Nice time in nice place! Close to city center! Good accomodation


{'roberta_neg': 0.0012416991,
 'roberta_neu': 0.012246823,
 'roberta_pos': 0.98651147}

In [69]:
res = {}
print("Running RoBERTa on all reviews. Grab a coffee, this might take a while...")

for i, row in tqdm(reviews.iterrows(), total=len(reviews)):
    
    text = str(row['comments']) 
    myid = row['id'] 
    
    try:
        encoded_text = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        output = model(**encoded_text)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        res[myid] = {
            'roberta_neg': scores[0],
            'roberta_neu': scores[1],
            'roberta_pos': scores[2]
        }
    except Exception as e:
        print(f"Error on ID {myid}: {e}")

roberta_df = pd.DataFrame(res).T
roberta_df = roberta_df.reset_index().rename(columns={'index': 'id'})
final_df = reviews.merge(roberta_df, how='left', on='id')

print("Done! Here is a peek at your new data:")
display(final_df.head())

Running RoBERTa on all reviews. Grab a coffee, this might take a while...


  0%|          | 0/5 [00:00<?, ?it/s]

Done! Here is a peek at your new data:


,listing_id,id,date,reviewer_id,reviewer_name,comments,city,unique_listing_id,roberta_neg,roberta_neu,roberta_pos
0,839610,33387684,2015-05-27,32412055,Giuseppe,Nice time in nice place! Close to city center!...,Amsterdam,Amsterdam_839610,0.001242,0.012247,0.986511
1,839610,33865462,2015-06-01,33521461,Chi Kwan,The host is very friendly and helpful,Amsterdam,Amsterdam_839610,0.001664,0.016924,0.981412
2,839610,47062612,2015-09-15,25521258,Scott,"Michael, Jacob's son was the one to coordinate...",Amsterdam,Amsterdam_839610,0.001249,0.006326,0.992425
3,839610,49847491,2015-10-06,32115691,Omer,Our stay was excelent. The appartment was clea...,Amsterdam,Amsterdam_839610,0.001145,0.007187,0.991668
4,839610,62280485,2016-02-13,16551781,Nina,it's very good! thanks! it's very beautiful al...,Amsterdam,Amsterdam_839610,0.001515,0.006005,0.992480


In [ ]:

raw_data_folder_reviews = r"C:\Users\DELL\Desktop\Programming\Data analytics\work\Airbnb\DataAirbnb\copy_data\reviews"

print("Reviews Pipeline Started. Scanning for files...\n")
inspector = inspect(engine)

for filename in os.listdir(raw_data_folder_reviews):
    # Make sure we only process Reviews files
    if filename.endswith(".csv") and "Reviews" in filename:
        file_path = os.path.join(raw_data_folder_reviews, filename)
        
        c = filename.find("Reviews")
        city = filename[:c]

        if inspector.has_table("reviews"):
            with engine.connect() as connection:
                # Check the REVIEWS table
                query = text(f"SELECT TOP 1 city FROM reviews WHERE city = '{city}'")
                result = connection.execute(query).fetchone()
                
                if result is not None:
                    print(f"[{city}] already exists in the Reviews table! Skipping.\n")
                    continue

        print(f"[{city}] New city found. Reading CSV...")
        df = pd.read_csv(file_path, low_memory=False)

        df['date'] = pd.to_datetime(df['date'])
        
        df['city'] = city
        
        df['unique_listing_id'] = city + "_" + df['listing_id'].astype(str)
        
        print(f'File {filename} is being pushed into SQL database....')
        df.to_sql("reviews", con=engine, if_exists='append', index=False, chunksize=100000)
        print(f'File {filename} successfully loaded.\n')

print("Pipeline Complete!")